In [1]:
using Pkg
Pkg.activate(".")

using Revise
using LinearAlgebra
using Base.Iterators
import DictionaryLearning.FoldyLax as FL
using Plots
using Base.Threads
using CUDA, Adapt, KernelAbstractions

include("../debug.jl")
;

  Activating project at `~/DictionaryLearning/examples/foldy-lax`


<!-- ## Settings & Setup -->

### Settings

In [2]:
# backend
if CUDA.functional()
    back = CUDABackend()
    println("$INF using Cuda")
else
    back = CPU()
    println("$INF using CPU")
end

# frequencies
f0 = 5e9        # central frequency
B = 1e9         # bandwidth
df = 40e6       # frequency resolution
λ = 3e8/f0

println("$INF central wavelength: $λ")

# recievers
r = zeros(3, 31)
r[1, :] .= -14.0
r[2, :] .= 0 .+ λ .* (-15 : 15)

# scatterers
ξ = zeros(3, 1500)
ξ[1,:] .= -12.0 .+ 10.0*rand(Float64, size(ξ, 2))
ξ[2,:] .= -5.0 .+ 10.0*rand(Float64, size(ξ, 2))

# scattering strength
τ = fill(0.3, size(ξ, 2))

# imaging window
iwx = 3e-1*λ .* (-90 : 90)
iwy = 2e-1*λ .* (-90 : 90)
z = zeros(3, length(iwx) * length(iwy))
for (i, (y, x)) in enumerate(Iterators.product(iwy, iwx))
    z[1,i] = x
    z[2,i] = y
end

println("$INF # scatterers: $(length(ξ))")
println("$INF # recievers: $(length(r))")
println("$INF # sources: $(length(z))")

[INF] using Cuda
[INF] central wavelength: 0.06
[INF] # scatterers: 4500
[INF] # recievers: 93
[INF] # sources: 98283


### Setup

In [ ]:
nfs = B÷df
fs = f0 .+ df .* (-nfs÷2 : nfs÷2)
p = length(fs)
n = size(r, 2)
m = size(ξ, 2)
k = size(z, 2)

# wave numbers
wns = fs .* (2π / 3e8);

println("$INF approx memory: $((
    16*(m*m*p + m*k*p + n*k*p + n*m*p + n*k*p + n*k*p) +
    8*max(m*m, m*k, n*k, n*m)
)*1e-9) GB")

# preallocate arrays
r = adapt(back, r)
ξ = adapt(back, ξ)
τ = adapt(back, τ)
z = adapt(back, z)
wns = adapt(back, collect(wns))
Mξξ = adapt(back, Array{ComplexF64}(undef, m, m, p))
Mξz = adapt(back, Array{ComplexF64}(undef, m, k, p))
Mrz = adapt(back, Array{ComplexF64}(undef, n, k, p))
Mrξ = adapt(back, Array{ComplexF64}(undef, n, m, p))
Gscat = adapt(back, Array{ComplexF64}(undef, n, k, p))
Ghom = adapt(back, Array{ComplexF64}(undef, n, k, p))

nws = max(m*m, m*k, n*k, n*m)
work = adapt(back, Vector{Float64}(undef, nws))

println("$INF actual memory: $(
    (sizeof(r) + sizeof(ξ) + sizeof(τ) + sizeof(z) +
    sizeof(wns) + sizeof(Mξξ) + sizeof(Mξz) + sizeof(Mrz) + sizeof(Mrξ) + 
    sizeof(Gscat) + sizeof(Ghom) + sizeof(work))*1e-9
) GB")

;

[INF] approx mem: 22.187041200000003 GB
[INF] actual memory: 22.187876408 GB


### Form & factor incidence matrices

In [ ]:
println("$STS solving for Mξξ")
FL.compM!(Mξξ, ξ, wns, work, τ)

println("$STS solving for Mξz")
FL.compM!(Mξz, ξ, z, wns, work)

println("$STS solving for Mrz")
FL.compM!(Mrz, r, z, wns, work)

println("$STS solving for Mrξ")
FL.compM!(Mrξ, r, ξ, wns, work, τ)

println("$STS factorizing Mξξ")
Mξξfac = [lu!(view(Mξξ, :, :, s)) for s in 1:p]
;

### Solve for Green's tensor

In [ ]:
# green's function for scattered system
FL.compG!(Gscat, Mξξfac, Mrξ, Mξz, Mrz, wns)

# green's function for homogenous system
Ghom = Mrz

;

<!-- ### Plot PSF -->

### Plot results

In [ ]:
function plotkm()
    # data matrix
    ky, kx = length(iwy), length(iwx)
    X = zeros(ComplexF64, kx, ky, p)
    x = view(X, fld(kx,2), fld(ky,2), :)
    x .= randn(ComplexF64, p)
    x ./= norm(x)
    X = adapt(back, X)

    Yscat = reshape(Gscat, n, k*p) * reshape(X, k*p)
    Yhom = reshape(Ghom, n, k*p) * reshape(X, k*p)

    kmscat = reshape(adjoint(reshape(Gscat, n, k*p)) * Yscat, k, p)
    kmhom = reshape(adjoint(reshape(Ghom, n, k*p)) * Yhom, k, p)

    imgscat = adapt(CPU(), reshape(sum(abs.(kmscat), dims=2), ky, kx))
    imghom = adapt(CPU(), reshape(sum(abs.(kmhom), dims=2), ky, kx))

    p1 = heatmap(iwx, iwy, imghom, title="Homogeneous", aspect_ratio = :equal, legend = false, axis=false)
    p2 = heatmap(iwx, iwy, imgscat, title="Scattered", aspect_ratio = :equal, legend = false, axis=false)
    return plot(p1, p2, layout=(1,2), size=(900, 400))
end

plotkm()